| Feature          | 의미                  | 값이 커질수록    |
| ---------------- | ------------------- | ---------- |
| `DSR단계`          | 차주단위 DSR 적용 단계      | 대출규제 강화    |
| `스트레스DSR단계`      | 스트레스 DSR 단계         | 대출규제 강화    |
| `규제지역여부`         | 해당 구가 규제지역인지        | 지역 규제 적용   |
| `주담대규제강화여부`      | 2025년 수도권 주담대 강화 상태 | 대출한도 제약    |
| `고가주택_주담대한도차등여부` | 고가주택 대출한도 차등화 상태    | 고가주택 구매 제약 |


In [1]:
# ============================================================
# 라이브러리 및 경로 설정
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


PROJECT_DIR = Path.cwd().parents[2]

DATA_DIR = PROJECT_DIR / "data"
EXTERNAL_DIR = DATA_DIR / "external"

V2_DATA_DIR = DATA_DIR / "v2_5year_seoul"
PROCESSED_DIR = V2_DATA_DIR / "processed"


TRADE_BASE_PATH = (
    PROCESSED_DIR
    / "df_fe_base.parquet"
)

POLICY_PATH = (
    EXTERNAL_DIR
    / "housing_policy_events_v1.csv"
)

POLICY_FEATURE_PATH = (
    PROCESSED_DIR
    / "df_fe_policy.parquet"
)


trade_base = pd.read_parquet(
    TRADE_BASE_PATH,
)

policy_raw = pd.read_csv(
    POLICY_PATH,
)


print("매매 기본:", trade_base.shape)
print("정책 원본:", policy_raw.shape)

매매 기본: (234302, 69)
정책 원본: (10, 10)


In [2]:
# ============================================================
# 정책 원본 구조 확인
# ============================================================

print(policy_raw.columns.tolist())

display(
    policy_raw[
        [
            "effective_date",
            "model_effective_month",
            "policy_category",
            "policy_name",
            "direction",
            "target_area",
        ]
    ]
)

['effective_date', 'model_effective_month', 'policy_category', 'policy_name', 'direction', 'target_area', 'policy_detail', 'source_org', 'source_url', 'v1_use']


,effective_date,model_effective_month,policy_category,policy_name,direction,target_area
0,2021-07-01,2021-07-01,DSR,차주단위 DSR 1단계,강화,서울 전역(당시 규제지역)
1,2022-01-01,2022-01-01,DSR,차주단위 DSR 2단계,강화,전국
2,2022-07-01,2022-07-01,DSR,차주단위 DSR 3단계,강화,전국
3,2023-01-05,2023-02-01,규제지역,서울 규제지역 대폭 해제,완화,서울 21개구
4,2024-02-26,2024-03-01,스트레스DSR,스트레스 DSR 1단계,강화,전국 은행권 주택담보대출
5,2024-09-01,2024-09-01,스트레스DSR,스트레스 DSR 2단계,강화,수도권 포함
6,2025-06-28,2025-07-01,주담대규제,수도권 가계부채 관리 강화,강화,서울 포함 수도권·규제지역
7,2025-07-01,2025-07-01,스트레스DSR,스트레스 DSR 3단계,강화,전국
8,2025-10-16,2025-11-01,규제지역,서울 전역 규제지역 확대 지정,강화,서울 25개구 전역
9,2025-10-16,2025-11-01,주담대규제,주택가격별 주담대 한도 차등화,강화,서울 포함 수도권·규제지역


In [4]:
# ============================================================
# 매매 데이터 지역 관련 컬럼 확인
# ============================================================

region_cols = [
    col
    for col in trade_base.columns
    if any(
        keyword in col
        for keyword in [
            "자치구",
            "시군구",
            "법정동",
            "구명",
        ]
    )
]

print(region_cols)
print()
print(
    trade_base[
        [
            "transaction_id",
            "거래월일자",
        ]
        + region_cols[:5]
    ].head()
)

['법정동시군구코드', '법정동읍면동코드', '법정동지번코드', '법정동본번코드', '법정동부번코드', '도로명시군구코드', '법정동', '시군구코드', '시군구명']

   transaction_id      거래월일자 법정동시군구코드  법정동읍면동코드  법정동지번코드  법정동본번코드  법정동부번코드
0               0 2021-07-01    11110     18700        1       82        0
1               1 2021-07-01    11110     11500        1        9        0
2               2 2021-07-01    11110     17900        1      199        0
3               3 2021-07-01    11110     11800        1       73        0
4               4 2021-07-01    11110     17400        1      703        0


In [5]:
# ============================================================
# 서울 자치구 확인
# ============================================================

print("자치구 수:", trade_base["시군구명"].nunique())

print(
    sorted(
        trade_base["시군구명"]
        .dropna()
        .unique()
    )
)

자치구 수: 25
['강남구', '강동구', '강북구', '강서구', '관악구', '광진구', '구로구', '금천구', '노원구', '도봉구', '동대문구', '동작구', '마포구', '서대문구', '서초구', '성동구', '성북구', '송파구', '양천구', '영등포구', '용산구', '은평구', '종로구', '중구', '중랑구']


In [6]:
# ============================================================
# 거래별 정책 상태 Feature 생성
# ============================================================

trade_policy = trade_base[
    [
        "transaction_id",
        "거래월일자",
        "시군구명",
    ]
].copy()

trade_policy["거래월일자"] = pd.to_datetime(
    trade_policy["거래월일자"]
)


# ------------------------------------------------------------
# 1. 차주단위 DSR 단계
# ------------------------------------------------------------

trade_policy["DSR단계"] = 0

trade_policy.loc[
    trade_policy["거래월일자"].ge("2021-07-01"),
    "DSR단계",
] = 1

trade_policy.loc[
    trade_policy["거래월일자"].ge("2022-01-01"),
    "DSR단계",
] = 2

trade_policy.loc[
    trade_policy["거래월일자"].ge("2022-07-01"),
    "DSR단계",
] = 3


# ------------------------------------------------------------
# 2. 스트레스 DSR 단계
# ------------------------------------------------------------

trade_policy["스트레스DSR단계"] = 0

trade_policy.loc[
    trade_policy["거래월일자"].ge("2024-03-01"),
    "스트레스DSR단계",
] = 1

trade_policy.loc[
    trade_policy["거래월일자"].ge("2024-09-01"),
    "스트레스DSR단계",
] = 2

trade_policy.loc[
    trade_policy["거래월일자"].ge("2025-07-01"),
    "스트레스DSR단계",
] = 3

In [7]:
# ============================================================
# 규제지역 여부
# ============================================================

CORE_REGULATED_GU = [
    "강남구",
    "서초구",
    "송파구",
    "용산구",
]

trade_policy["규제지역여부"] = 0


# 2021-07 ~ 2023-01
# 서울 전역 규제지역
mask_all_before = (
    trade_policy["거래월일자"]
    .between(
        "2021-07-01",
        "2023-01-01",
    )
)

trade_policy.loc[
    mask_all_before,
    "규제지역여부",
] = 1


# 2023-02 ~ 2025-10
# 강남·서초·송파·용산만 유지
mask_core = (
    trade_policy["거래월일자"]
    .between(
        "2023-02-01",
        "2025-10-01",
    )
    & trade_policy["시군구명"]
    .isin(CORE_REGULATED_GU)
)

trade_policy.loc[
    mask_core,
    "규제지역여부",
] = 1


# 2025-11 이후
# 서울 전역 다시 규제지역
mask_all_after = (
    trade_policy["거래월일자"]
    .ge("2025-11-01")
)

trade_policy.loc[
    mask_all_after,
    "규제지역여부",
] = 1

In [8]:
# ============================================================
# 주담대 규제 Feature
# ============================================================

trade_policy["주담대규제강화여부"] = (
    trade_policy["거래월일자"]
    .ge("2025-07-01")
    .astype(int)
)

trade_policy[
    "고가주택_주담대한도차등여부"
] = (
    trade_policy["거래월일자"]
    .ge("2025-11-01")
    .astype(int)
)

In [9]:
# ============================================================
# 정책 Feature 월별 검증
# ============================================================

policy_feature_cols = [
    "DSR단계",
    "스트레스DSR단계",
    "규제지역여부",
    "주담대규제강화여부",
    "고가주택_주담대한도차등여부",
]

monthly_policy_check = (
    trade_policy
    .groupby("거래월일자")
    .agg(
        DSR단계=("DSR단계", "first"),
        스트레스DSR단계=(
            "스트레스DSR단계",
            "first",
        ),
        규제지역_거래비율=(
            "규제지역여부",
            "mean",
        ),
        주담대규제강화여부=(
            "주담대규제강화여부",
            "first",
        ),
        고가주택_주담대한도차등여부=(
            "고가주택_주담대한도차등여부",
            "first",
        ),
    )
    .reset_index()
)

display(monthly_policy_check)

,거래월일자,DSR단계,스트레스DSR단계,규제지역_거래비율,주담대규제강화여부,고가주택_주담대한도차등여부
0,2021-07-01,1,0,1.000000,0,0
1,2021-08-01,1,0,1.000000,0,0
2,2021-09-01,1,0,1.000000,0,0
3,2021-10-01,1,0,1.000000,0,0
4,2021-11-01,1,0,1.000000,0,0
5,2021-12-01,1,0,1.000000,0,0
6,2022-01-01,2,0,1.000000,0,0
7,2022-02-01,2,0,1.000000,0,0
8,2022-03-01,2,0,1.000000,0,0
9,2022-04-01,2,0,1.000000,0,0


In [10]:
# ============================================================
# 월별 규제지역 자치구 수 검증
# ============================================================

gu_policy_check = (
    trade_policy[
        [
            "거래월일자",
            "시군구명",
            "규제지역여부",
        ]
    ]
    .drop_duplicates()
    .groupby("거래월일자")
    .agg(
        전체자치구=("시군구명", "nunique"),
        규제지역자치구=(
            "규제지역여부",
            "sum",
        ),
    )
    .reset_index()
)

display(gu_policy_check)

,거래월일자,전체자치구,규제지역자치구
0,2021-07-01,25,25
1,2021-08-01,25,25
2,2021-09-01,25,25
3,2021-10-01,25,25
4,2021-11-01,25,25
5,2021-12-01,25,25
6,2022-01-01,25,25
7,2022-02-01,25,25
8,2022-03-01,25,25
9,2022-04-01,25,25


In [11]:
# ============================================================
# 최종 정책 Feature 구성
# ============================================================

policy_feature_cols = [
    "DSR단계",
    "스트레스DSR단계",
    "규제지역여부",
    "주담대규제강화여부",
    "고가주택_주담대한도차등여부",
]

df_fe_policy = trade_policy[
    ["transaction_id"] + policy_feature_cols
].copy()

print("shape:", df_fe_policy.shape)

print(
    "transaction_id 중복:",
    df_fe_policy["transaction_id"]
    .duplicated()
    .sum(),
)

print("\n결측치:")
print(
    df_fe_policy[
        policy_feature_cols
    ].isna().sum()
)

shape: (234302, 6)
transaction_id 중복: 0

결측치:
DSR단계             0
스트레스DSR단계         0
규제지역여부            0
주담대규제강화여부         0
고가주택_주담대한도차등여부    0
dtype: int64


In [12]:
# ============================================================
# 정책 Feature 값 검증
# ============================================================

for col in policy_feature_cols:
    print(
        col,
        "→",
        sorted(
            df_fe_policy[col]
            .unique()
        ),
    )

DSR단계 → [np.int64(1), np.int64(2), np.int64(3)]
스트레스DSR단계 → [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
규제지역여부 → [np.int64(0), np.int64(1)]
주담대규제강화여부 → [np.int64(0), np.int64(1)]
고가주택_주담대한도차등여부 → [np.int64(0), np.int64(1)]


In [13]:
# ============================================================
# 정책 Feature 저장
# ============================================================

df_fe_policy.to_parquet(
    POLICY_FEATURE_PATH,
    index=False,
)

print("저장 완료:", POLICY_FEATURE_PATH)
print("최종 shape:", df_fe_policy.shape)

저장 완료: c:\Users\uk\Apartment price\data\v2_5year_seoul\processed\df_fe_policy.parquet
최종 shape: (234302, 6)
